# Does recent team history improve pre-match predictions?

A soccer analyst comparing Premier League matchups needs to know whether ten historical features improve on league-wide averages. One row is a completed fixture. The outcome is away win, draw or home win. Signed goal difference is home minus away goals.

The public-domain openfootball archive was retrieved on November 20, 2026. Its pinned source and checksums are recorded in the dataset card. From the repository root run `uv run python scripts/download_football.py` once. Read the adapter output to check season size, missingness, unique IDs and outcome proportions. Teams repeat throughout a season, so individual matches are dependent.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.experiments.football import run_validation

root = Path.cwd() if (Path.cwd() / 'configs').is_dir() else Path.cwd().parent
config = load_config(root / 'configs/football.toml')
matches = load_project_dataset(config).observations
display(matches.groupby('season').size().rename('fixtures'))
display(matches.isna().sum().sum())
display(matches.match_id.is_unique)
shares = pd.crosstab(matches.season, matches.outcome, normalize='index')
ax = shares.plot.bar(stacked=True, color=['#416b83', '#bb9b60', '#287a5a'])
ax.set(ylabel='Share of fixtures', xlabel='Premier League season')
plt.show()

Home wins form the largest class in each displayed season, so raw accuracy rewards a constant home-win decision. Elo, recent match count, and five-match scoring, conceding and points averages use previous dates only. `tests/test_football.py` changes current, same-date and future scores and compares features against unmodified inputs.

For standardized row $x_i$, ridge fits $b,x_i^T w$ by minimizing the sum of squared residuals plus $10\lVert w\rVert_2^2$. Shrinkage limits large coefficients when predictors overlap. Multinomial logistic maps three linear scores to probabilities using softmax and minimizes negative log probability of the observed class with an L2 penalty. Smaller $C$ gives stronger regularization. Both estimators use scikit-learn. The case study and integration are AI-assisted. Student estimator work is pending.

2018–19 initializes ratings. Each expanding fold fits parameters and preprocessing on seasons preceding validation. Validation features update once a date is completed. 2024–25 stays sealed. A training goal-difference mean and prior class frequencies are the regression and classification baselines. Compare models on MAE and log loss, with paired 28-day calendar block bootstrap intervals to retain some match dependence.

In [ ]:
report = run_validation(matches, config)['folds']
for season, fold in report.items():
    print(season, f"{fold['train_rows']} train / {fold['validation_rows']} validation")
    display(pd.DataFrame(fold['regression']).T[['mae', 'rmse', 'r2']].sort_values('mae'))
    display(pd.DataFrame(fold['classification']).T[['log_loss', 'balanced_accuracy', 'brier']].sort_values('log_loss'))
    print('Log-loss gain interval', fold['classification']['logistic_c_0_1']['gain_95_percent_block_interval'])
    print('MAE gain interval', fold['regression']['ridge_10']['gain_95_percent_block_interval'])

Ridge reduces validation MAE to roughly 1.37 goals in both seasons. Logistic with $C=0.1$ improves the frequency baseline in both seasons. This interval describes variation under the chosen time blocks, with limited support for dependence across blocks or a new season.

In the confusion matrix, a column of zeroes for predicted draws indicates a failed winner-only decision rule even when three-way log loss improves. Confidence checks compare the highest probability to observed correctness. They provide a limited view of calibration, so inspect draw-specific calibration before any promotion.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0.3, 1], [0.3, 1], color='0.5', linestyle='--', label='Perfect reliability')
for season, fold in report.items():
    model = fold['classification']['logistic_c_0_1']
    bins = pd.DataFrame(model['calibration'])
    ax.plot(bins.confidence, bins.accuracy, marker='o', label=season)
    display(pd.DataFrame(model['confusion'], index=model['class_order'], columns=model['class_order']))
ax.set(xlabel='Mean top-class probability', ylabel='Observed fraction correct', xlim=(0.3, 1), ylim=(0.3, 1))
ax.legend()
plt.show()

The high-confidence group was overconfident in 2022–23 and closer to observed accuracy in 2023–24. Similar Elo teams were harder in both folds. Only five matches per fold had fewer than five preceding archive games for at least one club, so the sparse-history slice carries little evidence. Carry both models as local reference demonstrations. Freeze student implementations, validation criteria and model selection before opening 2024–25.